In [44]:
import pyodbc
import os
from dotenv import load_dotenv
import pandas as pd

In [45]:
load_dotenv()

connection_string = (
    f"DRIVER={{{os.getenv('DB_DRIVER')}}};"
    f"SERVER={os.getenv('DB_SERVER')};"
    f"DATABASE={os.getenv('DB_DATABASE')};"
    f"Trusted_Connection={os.getenv('DB_TRUSTED_CONNECTION')};"
    f"TrustServerCertificate={os.getenv('DB_TRUST_SERVER_CERTIFICATE')};"
)


In [46]:
df = pd.read_csv("transformed_products_sql_ready.csv")
load_columns = [
    'id',
    'sku',
    'product_name',
    'category',
    'price',
    'currency',
    'stock',
    'rating',
    'in_stock',
    'supplier'
]

df_load = df[load_columns]
print(df_load.columns.tolist())
length = len(df_load)
print(length)


['id', 'sku', 'product_name', 'category', 'price', 'currency', 'stock', 'rating', 'in_stock', 'supplier']
147


In [47]:
try:
    connection = pyodbc.connect(connection_string)
    print("connected successfully")
except Exception as e:
    print("Connection error: ",e)

connected successfully


In [48]:
cursor = connection.cursor()

In [49]:
table_schema ="""BEGIN
    CREATE TABLE Products
    (
        id INT PRIMARY KEY,
        sku VARCHAR(50),
        product_name VARCHAR(150),
        category VARCHAR(100),
        price DECIMAL(10,2),
        currency VARCHAR(20),
        stock INT,
        rating DECIMAL(3,1),
        in_stock BIT,
        supplier VARCHAR(100),
    );
END
"""
cursor.execute(table_schema)
connection.commit()

ProgrammingError: ('42S01', "[42S01] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]There is already an object named 'Products' in the database. (2714) (SQLExecDirectW)")

In [50]:
batch_size = 20

query = """
INSERT INTO Products
(
    id,
    sku,
    product_name,
    category,
    price,
    currency,
    stock,
    rating,
    in_stock,
    supplier
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?);
"""

for start in range(0, length, batch_size):

    batch = df_load.iloc[start:start + batch_size]

    records = list(
        batch.itertuples(index=False, name=None)
    )

    try:

        cursor.executemany(query, records)

        connection.commit()

        print(f"Batch starting at {start} loaded successfully")

    except Exception as e:

        connection.rollback()

        print(f"Batch starting at {start} failed:", e)

Batch starting at 0 loaded successfully
Batch starting at 20 loaded successfully
Batch starting at 40 loaded successfully
Batch starting at 60 loaded successfully
Batch starting at 80 loaded successfully
Batch starting at 100 loaded successfully
Batch starting at 120 loaded successfully
Batch starting at 140 loaded successfully
